# Alchemy: commutativity and global assignment

Item names → scratch BPE → shared lexical roles → symmetric additive/residual composition → candidate similarity matrix → balanced square matching loss → Hungarian bijection.

**Điểm đã đo ở các version Kaggle trước:** R10 val5/30, Kaggle.0285/.1142, bằngR09. R11 lexical bilinear compatibility1/30, Kaggle0/0. Đây là mô hình để học matching, chưa phải lời giải mạnh.

**Bài học:** Ràng buộc đúng không tạo ra nghĩa của từ chưa thấy. 24/30 val results chưa xuất hiện ởtrain, dù0 UNK. Không tải BERT pretrained. RUN_ID9/10 so old loss và square matching; R11 kiểm tra giới hạn lexical attributes.

Notebook này chứa source đầy đủ nhưng chưa execute tại bản export này; execution_count=null và outputs rỗng là có chủ ý. Không suy diễn metric mới từ notebook chưa chạy.

Bật GPU T4 trên Kaggle, attach competition tương ứng trước Run All. Loader tự tìm cả `/kaggle/input/competitions/<slug>` và `/kaggle/input/<slug>`. Code assert T4, không tự chuyển CPU. Thực hành sau contest, không dùng trong thời gian cuộc thi cấm trợ giúp.


# Alchemy: thiếu kiến thức nghĩa và bài toán matching

## Bài khó ở đâu?

Chỉ có 150 quy tắc, 70 cặp test và 70 candidate result, mỗi result dùng một lần. Train fold có 120 quy tắc; val 30 tên kết quả tách rời. Cross-encoder scratch không có tri thức rằng hai nguyên tố kết hợp thành vật gì. Tăng depth/hard negatives không tạo ra tri thức thiếu. R09 shared roles + additive composition đạt val 5/30, Kaggle public .0285/private .1142. Val nhỏ, kết quả này chưa ổn định.

R06 Sinkhorn cũ chuẩn hóa minibatch 8 query ×16 candidate; không thể vừa có tổng mỗi hàng 1 và mỗi cột 1 trên ma trận chữ nhật ấy. Đây là mismatch objective với giả định bijection, không phải thử nghiệm sạch cho matching vuông. Hungarian chỉ tối ưu ma trận score đã có; score không mang nghĩa thì assignment vẫn sai.

## 5 paper ưu tiên

| Hạng | Paper / nguồn gốc | Phần phù hợp | Giới hạn / loại bỏ | Lượt |
|---|---|---|---|---|
| 1 | [Learning Latent Permutations with Gumbel-Sinkhorn Networks, Mena et al., ICLR 2018](https://arxiv.org/html/1802.08665v1) §2–3 | Relaxation trên ma trận vuông doubly stochastic | Không gọi normalization 8×16 là permutation; không thêm Gumbel noise khi không cần latent sampling | R10 balanced deterministic Sinkhorn |
| 2 | [Deep Sets, Zaheer et al., NeurIPS 2017](https://arxiv.org/html/1703.06114v3) §2–3 | Hàm ghép bất biến khi đổi item1/item2; shared item encoder | Symmetry chỉ bớt sample complexity, không tạo kiến thức vật lý | Giữ composer đối xứng R09 trong R10 |
| 3 | [An Embarrassingly Simple Approach to Zero-shot Learning, Romera-Paredes & Torr, ICML 2015](https://proceedings.mlr.press/v37/romera-paredes15.html) | Bilinear compatibility giữa feature query và thuộc tính candidate, ridge hai phía | Bài gốc có semantic attributes. BPE/character ở đây chỉ là lexical attributes, không tương đương semantic zero-shot | R11 kiểm tra giới hạn của lexical compatibility |
| 4 | [SuperGlue, Sarlin et al., CVPR 2020](https://arxiv.org/html/1911.11763v2) §3.2–3.3 | Học score cùng ràng buộc assignment thay vì sửa ở inference | Không dùng pretrained CV, geometry hoặc dustbin vì bài yêu cầu bijection đầy đủ | R10 row/column CE + Sinkhorn NLL |
| 5 | [Translating Embeddings for Modeling Multi-relational Data, Bordes et al., NeurIPS 2013](https://proceedings.neurips.cc/paper/5071-translating-embeddings-for-modeling-multi-relational-data.pdf) §2 | Shared entity space, low-capacity additive compatibility | item2 là entity, không phải relation như TransE; unseen disconnected names không thể dựa riêng entity ID | Giữ shared lexical roles; không gọi a+b là TransE reproduction |

## Thiết kế đóng băng trước scoring

**R10:** giữ nguyên composer R09. Minibatch chọn 16 result khác nhau và một rule cho mỗi result, tạo đúng score16×16. Row CE + column CE với inverse permutation, cộng .2×Sinkhorn NLL sau 20 log-normalization. Canary kiểm tổng hàng/cột≈1; assert target duy nhất. Cuối cùng vẫn Hungarian. 240 steps/180 giây, BPE train fold, cùng holdout và seed.

**R11:** fit BPE + TF-IDF character2–4 trên tên trong training fold, SVD≤64 chỉ fit training, query=[a+b, |a−b|, a*b, bias]. Giải bilinear compatibility bằng ridge hai phía alpha=beta=1 trên T4. Candidate chưa thấy vẫn có lexical attributes. Đây là phép thử xem chia sẻ hình thái có đủ không, không giả định tên gần chữ là gần nghĩa. Không grid search bằng 30 val hoặc leaderboard.

Cả hai in coverage: bao nhiêu tên val/test từng xuất hiện trong training ở bất kỳ vai trò nào, bao nhiêu từng làm input, số UNK. Không thêm cạnh heldout/label test, không tải BERT hay tokenizer pretrained. Chưa có điểm GPU vòng 2 tại thời điểm chốt thiết kế.


In [ ]:
SOURCE_COMMIT = '84e9a9efaa6346578162fa12915a8368f18e6485'
RUN_ID = 10


## Shared utilities

Seed, BPE, masking, data paths, schema checks và report writer.


In [ ]:
"""Utilities for controlled, idea-driven AICC NLP experiments."""
import ast, csv, hashlib, importlib.metadata, json, math, os, random, time
from copy import deepcopy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

SEED = 42
OUT = Path("/kaggle/working")
DEVICE = torch.device("cuda:0")


def start_run(task, run_id, config):
    assert torch.cuda.is_available(), "GPU T4 is required; do not silently fall back to CPU"
    name = torch.cuda.get_device_name(0)
    assert "T4" in name, "Expected T4, received " + name
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.set_num_threads(2)
    torch.backends.cudnn.benchmark = False
    torch.cuda.reset_peak_memory_stats()
    info = dict(task=task, run_id=f"R{run_id:02d}", config=config, seed=SEED,
                device=name, gpu_count=torch.cuda.device_count(), started=time.time(),
                source_commit=globals().get("SOURCE_COMMIT"),
                versions={p: importlib.metadata.version(p) for p in
                          ["torch", "numpy", "pandas", "scikit-learn", "tokenizers"]})
    print("RUN_START " + json.dumps(info), flush=True)
    return info


def find_data(slug):
    for root in (Path("/kaggle/input/competitions")/slug, Path("/kaggle/input")/slug):
        if root.is_dir():
            return root
    raise FileNotFoundError("Attach competition data: " + slug)


def make_bpe(texts, size=2048):
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    tokenizer.decoder = decoders.BPEDecoder()
    tokenizer.train_from_iterator(texts, trainers.BpeTrainer(
        vocab_size=size, min_frequency=2,
        special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]))
    return tokenizer


def encode_texts(tokenizer, texts, max_length):
    values = np.zeros((len(texts), max_length), dtype=np.int64)
    for i, encoded in enumerate(tokenizer.encode_batch(texts)):
        ids = encoded.ids[:max_length] or [1]
        values[i, :len(ids)] = ids
    return values


def hash_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def finish_run(info, validation, submission, checks, history, artifacts=None, limitations=None):
    assert checks.get("ids_unique") is True and checks.get("schema_valid") is True
    assert checks.get("row_count", 0) > 0
    submission.to_csv(OUT/"submission.csv", index=False)
    info.update(status="completed", elapsed_seconds=time.time()-info.pop("started"),
                validation=validation, submission_checks=checks, history=history,
                peak_gpu_memory_mb=torch.cuda.max_memory_allocated()/1024**2,
                artifacts=artifacts or {}, limitations=limitations or [],
                leaderboard_public=None, leaderboard_private=None)
    (OUT/"run_report.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
    print("NLP_RUN_RESULT " + json.dumps(info), flush=True)
    print("NLP_RUN_PASSED", flush=True)
    return info


class PositionEncoder(nn.Module):
    def __init__(self, vocab, dim=96, layers=2, max_length=384, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab, dim, padding_idx=0)
        self.position = nn.Embedding(max_length, dim)
        block = nn.TransformerEncoderLayer(dim, 4, dim*3, dropout=dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(block, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, ids):
        mask = ids.eq(0)
        x = self.embedding(ids) + self.position(torch.arange(ids.shape[1], device=ids.device))
        x = self.encoder(x, src_key_padding_mask=mask)
        return self.norm(x), mask


def pools(hidden, mask):
    valid = (~mask).unsqueeze(-1)
    mean = (hidden*valid).sum(1)/valid.sum(1).clamp(min=1)
    maximum = hidden.masked_fill(~valid, -1e4).amax(1)
    return torch.cat([mean, maximum], -1)



## Model và experiments

Các config Rxx nằm đầu source. Đổi RUN_ID trước khi Run All; mỗi lượt phải bắt đầu model/optimizer mới.


In [ ]:
"""Alchemy: scratch candidate-aware scoring with symmetry and global matching.
Architecture ideas adapted from the organizer's MIT-licensed BERT reference.
No pretrained weights or pretrained tokenizer are loaded.
"""
from scipy.optimize import linear_sum_assignment

ALCHEMY_CONFIGS={
 1:dict(idea="Scratch cross-encoder, swap consistency and Hungarian",hard=False,full=False,layers=2,probability=False,sinkhorn=False),
 2:dict(idea="Model-mined hard negatives with a random-negative mixture",hard=True,full=False,layers=2,probability=False,sinkhorn=False),
 3:dict(idea="CE over the complete training candidate list",hard=False,full=True,layers=2,probability=False,sinkhorn=False),
 4:dict(idea="Four scratch encoder layers test extra capacity",hard=False,full=False,layers=4,probability=False,sinkhorn=False),
 5:dict(idea="Average probabilities rather than logits of swapped inputs",hard=False,full=False,layers=2,probability=True,sinkhorn=False),
 6:dict(idea="Sinkhorn training models candidate competition before Hungarian",hard=False,full=False,layers=2,probability=False,sinkhorn=True),
 7:dict(idea="Combine hard negatives, full-list CE and probability averaging",hard=True,full=True,layers=2,probability=True,sinkhorn=False)
}


ALCHEMY_CONFIGS[8]={**ALCHEMY_CONFIGS[3],"idea":"Commutative three-tower composer with a shared scratch BPE embedding","composer":True}

ALCHEMY_CONFIGS[9]={**ALCHEMY_CONFIGS[8],"idea":"Tie input/result roles and use additive residual composition","tied_roles":True}
ALCHEMY_CONFIGS[10]={**ALCHEMY_CONFIGS[9],"idea":"Square bijective minibatches with row/column CE and balanced Sinkhorn","square":True,"papers":['1802.08665','1911.11763','1703.06114']}
ALCHEMY_CONFIGS[11]=dict(idea="ESZSL-inspired bilinear BPE and character attribute compatibility",papers=['pmlr-v37-romera-paredes15','5071-translating-embeddings'],alpha=1.,beta=1.)


def balanced_log_sinkhorn(matrix,iterations=20):
    assert matrix.ndim==2 and matrix.shape[0]==matrix.shape[1]
    logp=matrix.float()
    for _ in range(iterations):
        logp=logp-logp.logsumexp(1,keepdim=True)
        logp=logp-logp.logsumexp(0,keepdim=True)
    return logp

class AlchemyComposer(nn.Module):
    """Candidate embeddings share lexical features; item order is symmetric."""
    def __init__(self,vocab,dim=96,tied_roles=False):
        super().__init__()
        self.tied_roles=tied_roles
        self.embedding=nn.Embedding(vocab,dim,padding_idx=0)
        self.item=nn.Sequential(nn.Linear(dim,dim),nn.GELU(),nn.LayerNorm(dim))
        self.combine=nn.Sequential(nn.Linear(dim*3,dim*2),nn.GELU(),nn.Dropout(.1),nn.Linear(dim*2,dim))
        self.result=nn.Linear(dim,dim,bias=False)
        self.scale=nn.Parameter(torch.tensor(2.3))
    def forward(self,ids):
        segment=ids.eq(3).cumsum(1)
        valid=ids.gt(4)
        h=self.embedding(ids)
        vectors=[]
        for part in range(3):
            mask=(segment.eq(part)&valid).unsqueeze(-1)
            vectors.append((h*mask).sum(1)/mask.sum(1).clamp(min=1))
        a,b=self.item(vectors[0]),self.item(vectors[1])
        query=self.combine(torch.cat([a+b,(a-b).abs(),a*b],-1))
        if self.tied_roles:
            query=a+b+.1*query
            candidate=self.item(vectors[2])
        else:candidate=self.result(vectors[2])
        return (F.normalize(query,dim=-1)*F.normalize(candidate,dim=-1)).sum(-1)*self.scale.exp().clamp(max=30)


class AlchemyScorer(nn.Module):
    def __init__(self,vocab,layers):
        super().__init__()
        self.encoder=PositionEncoder(vocab,dim=96,layers=layers,max_length=64)
        self.head=nn.Sequential(nn.Linear(192,64),nn.GELU(),nn.Dropout(.1),nn.Linear(64,1))
    def forward(self,x):
        h,mask=self.encoder(x)
        return self.head(pools(h,mask)).squeeze(-1)


def run_alchemy(run_id):
    if run_id==11:return run_alchemy_attributes(run_id)
    cfg={**ALCHEMY_CONFIGS[run_id],"steps":240,"budget_seconds":180,"bpe_vocab":1024,"pretrained":False}
    info=start_run("alchemy",run_id,cfg)
    root=find_data("alchemy-aicc-round-4")
    all_train=pd.read_csv(root/"train.csv");test=pd.read_csv(root/"test.csv")
    candidates=sorted(pd.read_csv(root/"candidates.csv").result.unique().tolist())
    rng=np.random.default_rng(SEED)
    results=sorted(all_train.result.unique())
    held=set(rng.choice(results,size=max(8,len(results)//5),replace=False).tolist())
    # One row per held-out result makes the validation assignment genuinely bijective.
    validation=all_train[all_train.result.isin(held)].drop_duplicates('result').reset_index(drop=True)
    training=all_train[~all_train.result.isin(held)].reset_index(drop=True)
    train_candidates=sorted(training.result.unique())
    val_candidates=sorted(validation.result.unique())
    texts=training[['item1','item2','result']].astype(str).to_numpy().flatten().tolist()
    tokenizer=make_bpe(texts,cfg['bpe_vocab']);tokenizer.save(str(OUT/'tokenizer.json'))
    diagnostics=alchemy_diagnostics(training,validation,test,candidates,tokenizer)
    if cfg.get('square'):
        canary=balanced_log_sinkhorn(torch.arange(16,dtype=torch.float32,device=DEVICE).view(4,4)/16).exp()
        assert torch.allclose(canary.sum(0),torch.ones(4,device=DEVICE),atol=1e-4)
        assert torch.allclose(canary.sum(1),torch.ones(4,device=DEVICE),atol=1e-4)
    def triple(a,b,r):return '[CLS] '+str(a)+' [SEP] '+str(b)+' [SEP] '+str(r)+' [SEP]'
    def encode(rows,pool,swap=False):
        return torch.from_numpy(encode_texts(tokenizer,[triple(row.item2 if swap else row.item1,row.item1 if swap else row.item2,r)
                   for row in rows.itertuples() for r in pool],48))
    # Tokenization is deterministic and cached on CPU, not repeated in the training loop.
    tr_fwd=encode(training,train_candidates).view(len(training),len(train_candidates),48)
    tr_rev=encode(training,train_candidates,True).view_as(tr_fwd)
    val_fwd,val_rev=encode(validation,val_candidates),encode(validation,val_candidates,True)
    test_fwd,test_rev=encode(test,candidates),encode(test,candidates,True)
    model=(AlchemyComposer(tokenizer.get_vocab_size(),tied_roles=cfg.get("tied_roles",False)) if cfg.get('composer',False) else AlchemyScorer(tokenizer.get_vocab_size(),cfg['layers'])).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=.03)
    scaler=torch.amp.GradScaler('cuda')
    correct=torch.tensor([train_candidates.index(v) for v in training.result])
    val_gold=np.array([val_candidates.index(v) for v in validation.result])
    def scores(ids,grad=False):
        out=[]
        for batch in ids.split(128):
            with torch.autocast('cuda',dtype=torch.float16):out.append(model(batch.to(DEVICE)).float())
        return torch.cat(out)
    def assignment_metrics(matrix):
        _,hung=linear_sum_assignment(-matrix)
        available=set(range(matrix.shape[1]));greedy=[]
        for row in matrix:
            chosen=max(available,key=lambda j:row[j]);available.remove(chosen);greedy.append(chosen)
        return dict(hungarian_accuracy=float(np.mean(hung==val_gold)),independent_accuracy=float(np.mean(matrix.argmax(1)==val_gold)),
                    greedy_accuracy=float(np.mean(np.array(greedy)==val_gold)),duplicate_argmax_count=int(len(matrix)-len(set(matrix.argmax(1)))))
    @torch.inference_mode()
    def matrix(fwd,rev,n,m):
        model.eval();a=scores(fwd).view(n,m);b=scores(rev).view(n,m)
        if cfg['probability']:combined=(a.softmax(1)+b.softmax(1))/2
        else:combined=(a+b)/2
        return combined.cpu().numpy(),float((a-b).abs().mean())
    history=[];best=-1;best_state=None;hard_scores=None;tick=time.time()
    for step in range(cfg['steps']):
        model.train()
        if cfg.get('square'):
            chosen=rng.choice(len(train_candidates),size=min(16,len(train_candidates)),replace=False)
            idx=torch.tensor([int(rng.choice(np.flatnonzero(correct.numpy()==label))) for label in chosen])
        else:idx=torch.tensor(rng.choice(len(training),size=min(8,len(training)),replace=False))
        labels=correct[idx]
        pool=set(labels.tolist())
        if cfg.get('square'):pass  # exactly one query per candidate: genuine square bijection
        elif cfg['full']:pool=set(range(len(train_candidates)))
        else:
            if cfg['hard'] and hard_scores is not None:
                for i in idx.tolist():pool.update(hard_scores[i].argsort()[-3:].tolist())
            while len(pool)<min(16,len(train_candidates)):pool.add(int(rng.integers(len(train_candidates))))
        pool=torch.tensor(sorted(pool));targets=torch.tensor([pool.tolist().index(v) for v in labels.tolist()],device=DEVICE)
        fwd=tr_fwd[idx][:,pool].reshape(-1,48);rev=tr_rev[idx][:,pool].reshape(-1,48)
        optimizer.zero_grad(set_to_none=True)
        a=scores(fwd,True).view(len(idx),len(pool));b=scores(rev,True).view_as(a)
        combined=(a+b)/2
        loss=F.cross_entropy(combined,targets)+.05*F.mse_loss(a.softmax(1),b.softmax(1))
        if cfg.get('square'):
            assert len(idx)==len(pool) and targets.unique().numel()==len(pool)
            inverse=torch.argsort(targets)
            loss=.5*(F.cross_entropy(combined,targets)+F.cross_entropy(combined.T,inverse))
            logp=balanced_log_sinkhorn(combined)
            loss=loss-.2*logp[torch.arange(len(idx),device=DEVICE),targets].mean()
        if cfg['sinkhorn']:
            unique,counts=targets.unique(return_counts=True)
            if counts.max()==1:
                logp=combined
                for _ in range(5):
                    logp=logp-logp.logsumexp(1,keepdim=True)
                    logp=logp-logp.logsumexp(0,keepdim=True)
                loss=loss-.2*logp[torch.arange(len(idx),device=DEVICE),targets].mean()
        assert torch.isfinite(loss)
        scaler.scale(loss).backward();scaler.unscale_(optimizer);nn.utils.clip_grad_norm_(model.parameters(),1)
        scaler.step(optimizer);scaler.update()
        if (step+1)%40==0 or step==0:
            mat,asym=matrix(val_fwd,val_rev,len(validation),len(val_candidates));metrics=assignment_metrics(mat)
            record=dict(step=step+1,train_loss=float(loss.detach()),swap_gap=asym,**metrics)
            history.append(record);print('STEP '+json.dumps(record),flush=True)
            if metrics['hungarian_accuracy']>best:
                best=metrics['hungarian_accuracy'];best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            if cfg['hard']:
                with torch.inference_mode():hard_scores=scores(tr_fwd.reshape(-1,48)).view(len(training),-1).cpu().numpy()
            if time.time()-tick>cfg['budget_seconds']:break
    model.load_state_dict(best_state)
    val_matrix,asym=matrix(val_fwd,val_rev,len(validation),len(val_candidates))
    test_matrix,_=matrix(test_fwd,test_rev,len(test),len(candidates))
    _,assigned=linear_sum_assignment(-test_matrix)
    sub=pd.DataFrame({'Id':test.Id,'result':[candidates[i] for i in assigned]})
    np.save(OUT/'test_scores.npy',test_matrix);np.save(OUT/'validation_scores.npy',val_matrix)
    _,vp=linear_sum_assignment(-val_matrix)
    cases=validation.copy();cases['prediction']=[val_candidates[i] for i in vp];cases['correct']=vp==val_gold
    cases.to_csv(OUT/'validation_cases.csv',index=False)
    print('VALIDATION_CASES '+cases.head(12).to_json(orient='records'),flush=True)
    torch.save(best_state,OUT/'model.pt')
    checks=dict(ids_unique=sub.Id.is_unique,schema_valid=list(sub.columns)==['Id','result'],row_count=len(sub),
                results_unique=sub.result.is_unique,allowed_results=set(sub.result)==set(candidates))
    assert checks['results_unique'] and checks['allowed_results']
    return finish_run(info,dict(metric='bijective_holdout_accuracy',value=best,direction='maximize',samples=len(validation),
            split='fixed_result_disjoint_train_holdout',**assignment_metrics(val_matrix)),sub,checks,history,
            artifacts=dict(pretrained_weights=False,tokenizer='train-only BPE',validation_cases='validation_cases.csv',
                           training_rows=len(training),heldout_result_names=val_candidates,swap_gap=asym,coverage_diagnostics=diagnostics),
            limitations=['150 rules cannot replace BERT pretraining; organizer pretrained scores are a separate condition.',
                        'Validation is small and result-disjoint; single-seed differences are uncertain.',
                        'Checkpoint selection uses holdout Hungarian accuracy, not the Kaggle test leaderboard.',
                        'This checkpoint is trained on the training fold, without a final full-data refit.'])


def alchemy_diagnostics(training,validation,test,candidates,tokenizer):
    seen=set(training[['item1','item2','result']].astype(str).to_numpy().flatten())
    seen_inputs=set(training[['item1','item2']].astype(str).to_numpy().flatten())
    def stats(names):
        names=list(map(str,names));encoded=tokenizer.encode_batch(names)
        return dict(count=len(names),seen_any=sum(x in seen for x in names),seen_as_input=sum(x in seen_inputs for x in names),
                    unk_tokens=sum(e.ids.count(1) for e in encoded),total_tokens=sum(len(e.ids) for e in encoded))
    out=dict(training_rules=len(training),unique_train_names=len(seen),heldout_results=stats(validation.result),
             test_candidates=stats(candidates),test_input_names=stats(test[['item1','item2']].to_numpy().flatten()))
    print('ALCHEMY_COVERAGE '+json.dumps(out),flush=True)
    return out


def run_alchemy_attributes(run_id):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    from scipy.sparse import hstack
    cfg={**ALCHEMY_CONFIGS[run_id],"pretrained":False,"bpe_vocab":1024,"attribute_dim":64}
    info=start_run('alchemy',run_id,cfg);root=find_data('alchemy-aicc-round-4')
    train=pd.read_csv(root/'train.csv');test=pd.read_csv(root/'test.csv')
    candidates=sorted(pd.read_csv(root/'candidates.csv').result.unique().tolist())
    rng=np.random.default_rng(SEED);results=sorted(train.result.unique())
    held=set(rng.choice(results,size=max(8,len(results)//5),replace=False).tolist())
    validation=train[train.result.isin(held)].drop_duplicates('result').reset_index(drop=True)
    training=train[~train.result.isin(held)].reset_index(drop=True)
    names=sorted(set(training[['item1','item2','result']].astype(str).to_numpy().flatten()))
    tokenizer=make_bpe(names,cfg['bpe_vocab']);tokenizer.save(str(OUT/'tokenizer.json'))
    def pieces(items):return [' '.join(map(str,tokenizer.encode(str(x)).ids)) for x in items]
    bpe=TfidfVectorizer(token_pattern=r'\d+',lowercase=False)
    chars=TfidfVectorizer(analyzer='char',ngram_range=(2,4),max_features=2048,sublinear_tf=True)
    fit=hstack([bpe.fit_transform(pieces(names)),chars.fit_transform(names)]).tocsr()
    svd=TruncatedSVD(n_components=min(64,len(names)-1,fit.shape[1]-1),random_state=SEED).fit(fit)
    def attributes(items):
        items=list(map(str,items));x=svd.transform(hstack([bpe.transform(pieces(items)),chars.transform(items)]).tocsr()).astype(np.float32)
        x/=np.maximum(np.linalg.norm(x,axis=1,keepdims=True),1e-6)
        return torch.from_numpy(x).to(DEVICE)
    def queries(rows):
        a,b=attributes(rows.item1),attributes(rows.item2)
        return torch.cat([a+b,(a-b).abs(),a*b,torch.ones((len(rows),1),device=DEVICE)],1)
    pool=sorted(training.result.unique());S=attributes(pool);X=queries(training)
    target=attributes(training.result)
    # ESZSL-style two-sided Tikhonov compatibility, lexical attributes replace semantic attributes.
    left=X.T@X+cfg['alpha']*torch.eye(X.shape[1],device=DEVICE)
    right=S.T@S+cfg['beta']*torch.eye(S.shape[1],device=DEVICE)
    W=torch.linalg.solve(left,X.T@target)
    W=torch.linalg.solve(right,W.T).T
    assert torch.isfinite(W).all()
    def scores(rows,pool):return (queries(rows)@W@attributes(pool).T).cpu().numpy()
    val_pool=sorted(validation.result.unique());gold=np.array([val_pool.index(x) for x in validation.result])
    matrix=scores(validation,val_pool);_,assigned=linear_sum_assignment(-matrix)
    metric=float(np.mean(assigned==gold));test_scores=scores(test,candidates);_,prediction=linear_sum_assignment(-test_scores)
    sub=pd.DataFrame(dict(Id=test.Id,result=[candidates[i] for i in prediction]))
    cases=validation.copy();cases['prediction']=[val_pool[i] for i in assigned];cases['correct']=assigned==gold
    cases.to_csv(OUT/'validation_cases.csv',index=False)
    np.save(OUT/'test_scores.npy',test_scores);np.save(OUT/'validation_scores.npy',matrix)
    torch.save(dict(W=W.cpu(),source_commit=info['source_commit']),OUT/'compatibility.pt')
    diagnostic=alchemy_diagnostics(training,validation,test,candidates,tokenizer)
    print('VALIDATION_CASES '+cases.to_json(orient='records'),flush=True)
    return finish_run(info,dict(metric='bijective_holdout_accuracy',value=metric,direction='maximize',samples=len(validation),split='fixed_result_disjoint_train_holdout',
        independent_accuracy=float(np.mean(matrix.argmax(1)==gold)),duplicate_argmax_count=int(len(matrix)-len(set(matrix.argmax(1))))),sub,
        dict(ids_unique=sub.Id.is_unique,schema_valid=list(sub.columns)==['Id','result'],row_count=len(sub),results_unique=sub.result.is_unique,finite=bool(np.isfinite(test_scores).all())),[],
        artifacts=dict(pretrained_weights=False,tokenizer='train-only BPE plus character attribute features',coverage_diagnostics=diagnostic,training_rows=len(training),heldout_result_names=val_pool),
        limitations=['ESZSL-inspired compatibility, not a semantic zero-shot reproduction: character/BPE attributes contain no external world knowledge.',
                    'The training-fold SVD and vocabulary never fit held-out or test names.',
                    'A 30-rule holdout has high variance; fixed regularizers were chosen before scoring.'])


In [ ]:
OUT = Path('/kaggle/working') / 'alchemy' / f'R{RUN_ID:02d}'
OUT.mkdir(parents=True, exist_ok=True)
report = run_alchemy(RUN_ID)
print('Submission:', OUT / 'submission.csv')
